# Classify, route and evaluate with Drex in Haystack

[Haystack](https://github.com/deepset-ai/haystack) is deepset’s open-source Python framework for AI pipelines and agents. This Nace-maintained integration adds Drex components. See the [Haystack documentation](https://docs.haystack.deepset.ai/docs/intro) for the framework’s concepts.

[Drex](https://console.nace.ai/docs) is Nace's family of decision models. A Drex model doesn't generate text. You send it a text (or JSON, or images on Drex Max) with typed questions, and it answers all of them in one request with calibrated probabilities:

| Question type | What it asks | What Drex returns |
| --- | --- | --- |
| `choice` | Which of these labels fits? | The picked label, its confidence, and a probability for every label |
| `score` | Where does this fall on these ordered levels? | A score from 0 to the top level, with a probability for every level |
| `noul` | Is this statement true? | The probability that it is |

The `nace-haystack` package brings Drex to Haystack pipelines. This notebook uses three of its components on eight synthetic support emails:

1. **Index.** `NaceDocumentClassifier` answers three questions about each email once and stores the answers in the document's metadata, where metadata filters can use them.
2. **Route.** `NaceTextRouter` sends each email to a team, with a `low_confidence` fallback. One email has a scanned invoice attached, and Drex Max reads the image together with the text.
3. **Evaluate.** `NaceFaithfulnessEvaluator` checks two draft replies against a refund policy.

The last section shows the edits that switch an existing TypeSafe (Jev) pipeline to Drex.

All data sent to Drex is synthetic, and a full run uses a few cents of usage at most.

## Setup

**Install.** `nace-haystack` is [published on PyPI](https://pypi.org/project/nace-haystack/). Use Python 3.11 or later and a checkout of [this repository](https://github.com/nace-ai/nace-haystack) for the sample emails and invoice generator. From the repository root, in an activated virtual environment:

```bash
python -m pip install nace-haystack pillow jupyterlab
export NACE_API_KEY="your-nace-api-key"
python -m jupyterlab notebooks/nace_haystack_cookbook.ipynb
```

Select that environment’s Python kernel. The optional install line in the next cell uses the published package. Pillow draws and shrinks the synthetic invoice. Each model call consumes credit; see [pricing](https://console.nace.ai/docs/drex/reference/pricing).

**API key.** The components read the key from the `NACE_API_KEY` environment variable. Create a key in the [Nace console](https://console.nace.ai) and export it before you start Jupyter. Never paste a key into a notebook: it would be saved in the file along with the outputs.

**Model.** This notebook pins `MODEL = "drex-max-v1.1"`, which reads text, JSON and images. The default moves to the newest Drex model in package releases, and a new model can shift probabilities, so pin `model=` in production if your pipeline relies on thresholds like the ones below.

In [ ]:
# Optional, if the selected kernel does not have the packages installed:
# %pip install nace-haystack pillow

import os

MODEL = "drex-max-v1.1"

if not os.environ.get("NACE_API_KEY"):
    msg = "Set the NACE_API_KEY environment variable before you start Jupyter."
    raise RuntimeError(msg)
print("NACE_API_KEY is set.")

## Load the emails

`examples/data/emails.json` holds eight synthetic support emails. The names, numbers and addresses are made up. Two emails ask for a refund (e1 and e2). The others ask about a tax line (e3), report problems in the product (e4 to e6), or ask about pricing and a demo (e7 and e8). Email e1 comes with a scanned invoice, which we use in the routing section.

In [ ]:
import json
from pathlib import Path

# This notebook sits in notebooks/, next to examples/. Run it from the repository root or from notebooks/.
REPO = Path.cwd() if (Path.cwd() / "examples").exists() else Path.cwd().parent
EXAMPLES = REPO / "examples"
DATA = EXAMPLES / "data"

emails = json.loads((DATA / "emails.json").read_text())
for email in emails:
    attachment = f"   [attachment: {email['attachment']}]" if email["attachment"] else ""
    print(f"{email['id']}  {email['subject']}{attachment}")

## Index: classify every email once

`NaceDocumentClassifier` asks Drex typed questions about each document. All the questions for a document go in one request, and the answers are stored in `document.meta["nace"]`, keyed by question ID.

We ask three questions about every email:

- `department` is a `choice` question with three labels. Each label has a description that says what it covers.
- `urgency` is a `score` question with three ordered levels, from `can wait` (0) to `today` (2).
- `refund` is a `noul` question: the probability that the customer is asking for a refund.

In the indexing pipeline, the classifier feeds a `DocumentWriter`, which writes the classified documents to an in-memory document store. Documents that can't be classified come out of the classifier's `failed_documents` output instead, and the pipeline's result below lists them (none here).

In [ ]:
from haystack import Document, Pipeline
from haystack.components.writers import DocumentWriter
from haystack.document_stores.in_memory import InMemoryDocumentStore

from haystack_integrations.components.classifiers.nace import NaceDocumentClassifier

QUESTIONS = {
    "department": {
        "type": "choice",
        "instructions": "Which department should handle this email?",
        "criteria": {
            "billing": "Invoices, payments, refunds and taxes",
            "technical": "Crashes, errors, sign-in problems and bugs",
            "sales": "Pricing, plans, demos and new purchases",
        },
    },
    "urgency": {
        "type": "score",
        "instructions": "How urgent is this email?",
        "criteria": ["can wait", "this week", "today"],
    },
    "refund": {"type": "noul", "instructions": "Is the customer asking for a refund?"},
}

store = InMemoryDocumentStore()
indexing = Pipeline()
indexing.add_component("classifier", NaceDocumentClassifier(model=MODEL, questions=QUESTIONS))
indexing.add_component("writer", DocumentWriter(document_store=store))
indexing.connect("classifier.documents", "writer.documents")

documents = [
    Document(content=f"Subject: {email['subject']}\n\n{email['body']}", meta={"email_id": email["id"]})
    for email in emails
]
indexing_result = indexing.run({"classifier": {"documents": documents}})
print(indexing_result)

In [ ]:
import pprint


def rounded(value, digits=2):
    """Round every float in a nested answer, so that it prints readably."""
    if isinstance(value, float):
        return round(value, digits)
    if isinstance(value, dict):
        return {key: rounded(item, digits) for key, item in value.items()}
    if isinstance(value, list):
        return [rounded(item, digits) for item in value]
    return value


indexed = sorted(store.filter_documents(), key=lambda document: document.meta["email_id"])

print(f"{'email':<6}{'department':<12}{'conf.':<7}{'urgency':<9}{'refund':<8}subject")
for document in indexed:
    answers = document.meta["nace"]
    department = answers["department"]
    subject = document.content.splitlines()[0].removeprefix("Subject: ")
    print(
        f"{document.meta['email_id']:<6}{department['choice']:<12}{department['confidence']:<7.2f}"
        f"{answers['urgency']['score']:<9.1f}{answers['refund']['noul']:<8.2f}{subject}"
    )
print("\nurgency runs from 0 (can wait) to 2 (today). refund is the probability that the email asks for one.")

print("\nEverything stored with e1, rounded for display:")
pprint.pprint(rounded(indexed[0].meta["nace"]), sort_dicts=False)

The answers are nested metadata, so Haystack's metadata filters can read them. The refund probability of each email is at `meta.nace.refund.noul`. A filter on it finds the refund requests without another call to Drex:

In [ ]:
refund_requests = store.filter_documents(
    filters={"field": "meta.nace.refund.noul", "operator": ">", "value": 0.5},
)
print("Refund requests:", sorted(document.meta["email_id"] for document in refund_requests))

## Route: send each email to a team

`NaceTextRouter` asks Drex one `choice` question and sends the text to the output of the label the model picks. The router below has:

- `labels`: one output per team. A dict gives each label a description.
- `min_confidence=0.6`: when the confidence of the answer is below this, the text goes to a `low_confidence` output instead. A fallback, such as an LLM or a person, can take it from there.
- `include_probabilities=True`: a `probabilities` output carries the full answer (the picked label, its confidence and every label's probability), so thresholds can live in your pipeline too.

`run` returns a dict with the text under the chosen output, plus `probabilities`. Here are two texts. The first is clearly about billing. The second says almost nothing, so Drex isn't confident about any team.

The 0.6 threshold is illustrative: choose it on labeled examples from your own workload. Low confidence routes to review; API errors still raise. The full connected example is [`examples/route_with_fallback.py`](../examples/route_with_fallback.py).


In [ ]:
import pprint

from haystack_integrations.components.routers.nace import NaceTextRouter

TEAMS = {
    "billing": "Invoices, payments, refunds and taxes",
    "technical": "Crashes, errors, sign-in problems and bugs",
    "sales": "Pricing, plans, demos and new purchases",
}

router = NaceTextRouter(
    model=MODEL,
    labels=TEAMS,
    instructions="Which team should handle this email? Image 1, if present, is the customer's attachment.",
    min_confidence=0.6,
    include_probabilities=True,
)

for text in [
    "I was charged twice for my subscription.",
    "Hello, I have a question about my account. Could someone get back to me? Thanks, Pat",
]:
    pprint.pprint(rounded(router.run(text=text)), sort_dicts=False, width=110)
    print()

Now the whole inbox. Email e1 has a scanned invoice attached, and the router's instructions tell the model that Image 1 is the customer's attachment.

The scan is a phone-sized photo: 3024 x 4032 pixels and about 5.5 MB, over the 4 MiB Drex accepts in one request. `nace-haystack` shrinks an image that doesn't fit and logs a warning that says so. The shrunk image still carries what the model reads, because Drex Max works on images at about 1.7 megapixels. To skip the shrinking, downscale the image yourself, for example with `ImageContent.from_file_path(path, size=(2048, 2048))`.

The next cell loads the scan, drawing it with `make_invoice()` from the examples folder if the file is missing, and shows it at preview size.

In [ ]:
import sys

import PIL.Image
from haystack.dataclasses import ImageContent
from IPython.display import display

sys.path.insert(0, str(EXAMPLES))  # makes examples/make_invoice.py importable
from make_invoice import make_invoice


def scanned_invoice() -> Path:
    path = DATA / "scanned_invoice.png"
    if not path.exists():
        make_invoice().save(path)
    return path


scan = PIL.Image.open(scanned_invoice())
print(f"Scanned invoice: {scan.width} x {scan.height} pixels, {scanned_invoice().stat().st_size / 1e6:.1f} MB")

preview = scan.convert("RGB")
preview.thumbnail((420, 560))
display(preview, include=["image/jpeg"])  # JPEG only: a PNG would keep the grain and bloat the notebook

In [ ]:
for email in emails:
    images = [ImageContent.from_file_path(scanned_invoice())] if email["attachment"] else None
    routed = router.run(text=f"Subject: {email['subject']}\n\n{email['body']}", images=images)
    team = next(output for output in routed if output != "probabilities")
    answer = routed["probabilities"]
    note = " (with the scanned invoice)" if images else ""
    print(f"{email['id']}  -> {team:<14} confidence {answer['confidence']:.2f}{note}")

The warning in the output above says the scan was shrunk to fit one request.

Drex Max can also answer questions about the image itself. A second router reads the total on the scan. Its labels are three possible totals, and the text it gets doesn't mention an amount, so the answer has to come from Image 1. Only one label is right: the scan's total due is $4,212.00.

In [ ]:
invoice_reader = NaceTextRouter(
    model=MODEL,
    labels={"total_4212": "$4,212.00", "total_2106": "$2,106.00", "total_1042": "$1,042.00"},
    instructions="What total amount is due on the invoice in Image 1?",
    include_probabilities=True,
)
total = invoice_reader.run(text="The customer's attachment.", images=[ImageContent.from_file_path(scanned_invoice())])
answer = total["probabilities"]
print(f"Total read from the scan: {invoice_reader.labels[answer['choice']]} (confidence {answer['confidence']:.2f})")

## Evaluate: check replies against the policy

Before a drafted reply goes out, you can check it against the policy it should follow. `NaceFaithfulnessEvaluator` has the same inputs (`questions`, `contexts`, `predicted_answers`) and outputs (`score`, `individual_scores`, `results`, `meta`) as Haystack's `FaithfulnessEvaluator`. It scores differently: each item is one Drex request, and the score is the probability that every claim in the answer is supported by the contexts.

Below, two draft replies to e1 are checked against the refund policy. The first restates the policy for duplicate charges. The second offers a 50% credit that the policy doesn't mention. The package also has `NaceContextRelevanceEvaluator`, which judges whether retrieved contexts help answer the question.

In [ ]:
from haystack_integrations.components.evaluators.nace import NaceFaithfulnessEvaluator

REFUND_POLICY = (
    "Duplicate charges are refunded in full within 5 business days. "
    "Annual plans cancelled within 30 days of renewal get a prorated refund. Other fees are not refundable."
)

evaluator = NaceFaithfulnessEvaluator(model=MODEL)
result = evaluator.run(
    questions=[
        "What happens to the duplicate charge on INV-1042?",
        "What happens to the duplicate charge on INV-1042?",
    ],
    contexts=[[REFUND_POLICY], [REFUND_POLICY]],
    predicted_answers=[
        "The duplicate charge will be refunded in full within 5 business days.",
        "We can offer a 50% credit on your next invoice instead of a refund.",
    ],
)
for name, score in zip(["policy-backed reply", "made-up offer"], result["individual_scores"], strict=True):
    print(f"{name:<20} faithfulness {score:.2f}")

## Switching from TypeSafe

The router and classifier mirror deepset's `typesafe-haystack` components for Jev. They take the same parameters and produce the same outputs as `TypeSafeTextRouter` and `TypeSafeDocumentClassifier`, so a Jev pipeline switches to Drex with a few edits.

In Python code that uses Jev's default model, change the import, the class name and the API key:

```diff
-from haystack_integrations.components.routers.typesafe import TypeSafeTextRouter
+from haystack_integrations.components.routers.nace import NaceTextRouter

-router = TypeSafeTextRouter(labels=..., min_confidence=0.6)
+router = NaceTextRouter(labels=..., min_confidence=0.6)
```

```diff
-export TYPESAFE_API_KEY=...
+export NACE_API_KEY=...
```

In a saved YAML pipeline, make three edits per component:

```diff
-    type: haystack_integrations.components.routers.typesafe.text_router.TypeSafeTextRouter
+    type: haystack_integrations.components.routers.nace.text_router.NaceTextRouter
...
-        - TYPESAFE_API_KEY
+        - NACE_API_KEY
...
-      model: jev-latest
+      model: drex-max-v1.1
```

The classifier's type becomes `haystack_integrations.components.classifiers.nace.document_classifier.NaceDocumentClassifier`. A saved pipeline keeps `metadata_field: typesafe`, so answers stay where your filters read them. Model names aren't translated: Drex rejects `jev-latest` with a message listing its models.

Drex's own guide covers the API side: [Migrate from TypeSafe](https://console.nace.ai/docs/drex/guides/migrate-from-typesafe).